In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
import numpy as np


# 1.自定义Transformer架构
class TransformerPositional(keras.layers.Layer):

    def __init__(self, vocab_size, sequence_length, embed_dim, **kwargs):
        super().__init__(**kwargs)

        self.vocab_size = vocab_size
        self.sequence_length = sequence_length
        self.embed_dim = embed_dim

        self.token_emd = keras.layers.Embedding(
            input_dim=self.vocab_size,
            output_dim=self.embed_dim,
        )

        self.pos_emd = keras.layers.Embedding(
            input_dim=self.sequence_length,
            output_dim=self.embed_dim,
        )

    def call(self, inputs):

        token_embed = self.token_emd(inputs)

        pos = keras.ops.arange(self.sequence_length)
        pos_embed = self.pos_emd(pos)
        pos_embed = keras.ops.expand_dims(pos_embed, axis=0)

        return token_embed + pos_embed

    def get_config(self):

        config = super().get_config()
        config.update(
            {
                "vocab_size": self.vocab_size,
                "sequence_length": self.sequence_length,
                "embed_dim": self.embed_dim,
            }
        )


class TransformerDecoder(keras.layers.Layer):

    def __init__(self, embed_dim, dense_dim, num_heads, dropout=0.1, **kwargs):
        super().__init__(**kwargs)

        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads

        self.attention = keras.layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.embed_dim // num_heads,
        )

        self.ffn = keras.Sequential(
            [
                keras.layers.Dense(units=self.dense_dim, activation="relu"),
                keras.layers.Dense(
                    units=self.embed_dim,
                ),
            ]
        )

        self.dropout1 = keras.layers.Dropout(dropout)
        self.dropout2 = keras.layers.Dropout(dropout)

        self.norm1 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs, training=False):

        attn_out = self.attention(
            query=inputs,
            key=inputs,
            value=inputs,
            use_causal_mask=True,
        )
        attn_out = self.dropout1(attn_out, training=training)

        out1 = self.norm1(attn_out + inputs)

        ffn_out = self.ffn(out1)
        ffn_out = self.dropout2(ffn_out, training=training)

        return self.norm2(out1 + ffn_out)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
            }
        )


def build_model():

    print("正在搭建模型..........")

    vocab_size = 30
    max_len = 5
    embed_dim = 32

    inputs = keras.Input(shape=(max_len,), dtype="int32")

    x = TransformerPositional(
        vocab_size=vocab_size,
        sequence_length=max_len,
        embed_dim=embed_dim,
    )(inputs)

    x = TransformerDecoder(
        embed_dim=embed_dim,
        dense_dim=64,
        num_heads=2,
    )(x)

    x = x[:, -1, :]

    outputs = keras.layers.Dense(units=vocab_size, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer="adam",
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )

    print("正在生成数据..........")

    x_train, y_train = [], []

    for start in range(0, 15):
        seq = [start + i * 2 for i in range(max_len + 1)]

        x_train.append(seq[:-1])
        y_train.append(seq[-1])

    x_train = np.array(x_train)
    y_train = np.array(y_train)

    print("开始训练模型.......")

    model.fit(x_train, y_train, epochs=150)

    print("模型训练完毕.........")

    return model


def main():

    print("主程序开始启动.........")

    model = build_model()

    test_seq = np.array([[1, 3, 5, 7, 9]])

    pred_probs = model.predict(test_seq, verbose=1)

    pred_num = np.argmax(pred_probs)
    
    print("开始预测数据.........")

    print("\n" + "=" * 40)
    print(f"输入序列: [1, 3, 5, 7, 9]")
    print(f"模型预测的下一个数字是: ** {pred_num} **")
    print(f"预测 {pred_num} 的置信度/概率: {np.max(pred_probs)*100:.2f}%")
    print("=" * 40)


if __name__ == "__main__":

    main()